# 00 · Summary: Victorian rental prices

**MAST30034 Project 2, Group 55.** This notebook is the summary of the whole project for clients and stakeholders.
It covers the question we were asked, the answers, how we got them, the problems we ran into, and the assumptions
and limits to keep in mind. The detailed work is in notebooks `01` to `07` (see the map at the end).

It only reads the summary tables in `data/curated/` and the charts in `plots/`, which are all committed to the
repository, so it runs in a few seconds without any raw data.

**The brief.** A real estate company wants to know (1) what rent to list its properties at and (2) which properties
are most likely to rise in rent over the next five years. For Victoria, that means three questions:

1. What are the most important internal and external features in predicting rental prices?
2. What are the top 10 suburbs with the highest predicted growth rate?
3. What are the most liveable and affordable suburbs according to our chosen metrics?

## The answers in short

**1. What drives rent.** The **size of the home** matters most: bedrooms, bathrooms and car spaces. Among the
features of the location, **access to the city** matters most, then the **area's income** and **dining and
entertainment nearby**. In Melbourne, each extra bedroom adds about **\$98 a week** and each bathroom about
**\$81**. Being furnished, a study, a dishwasher and a balcony add **\$14 to \$48**. The same plain 2-bedroom
apartment is worth about \$618 a week in Southbank, \$556 in Box Hill and \$396 in Shepparton.

**2. Where rents will grow fastest to 2030.** Mostly **regional centres**: Swan Hill, Moe–Newborough, Warragul,
Morwell, Mildura, Shepparton, Wodonga and Benalla, plus **Frankston** and **Croydon–Lilydale** in Melbourne, at
about 6.4–8.2% a year. Growth overall should be slower than the post-COVID surge: about 3% a year for a typical
Melbourne area and 5% for a typical regional one.

**3. Liveable and affordable.** Inner Melbourne is the most liveable (Melbourne CBD, Carlton, West Melbourne,
Fitzroy, Collingwood), but rent there takes a large share of local incomes. 77 Melbourne suburbs and 6 regional
suburbs are both more liveable and more affordable than the median; the best mix is **Collingwood** (5th most
liveable, 11th most affordable) and, in regional Victoria, **Geelong West** (10th most affordable). For investors,
**50 suburbs** are both liveable and above the median forecast growth. **Fairfield** and **Geelong West** combine both
with low rents, and **Fairfield** also ranks first on our weighted score.

## 1. The data

| Data | Source | Used for |
|---|---|---|
| Rental listings, September 2025 | Domain.com.au (course dataset) | 12,292 usable listings |
| Rental listings, September 2026 | rent.com.au, scraped by the group (`02`) | 9,157 usable listings |
| Rent history 2000–2025 by area | DFFH Rental Report (median rent of new bonds, quarterly) | Q2 forecast |
| SA2 boundaries, population 2001–25, personal income 2022–23 | ABS | Area features, affordability |
| Population projections to 2036 | Victoria in Future 2023 (VIF2023) | Area features |
| Train, tram and bus stops | PTV GTFS timetable | Transport access |
| Schools | Victorian Department of Education | Education access |
| Parks, shops, hospitals, entertainment | OpenStreetMap | Amenity access |
| Driving distances to the CBD and nearest station | OSRM (OpenStreetMap roads) | City access |

In total there are **21,449 usable listings** (two snapshots, a year apart) in about 650 suburbs. **99.8% of all
listings (21,861 of 21,895) are placed in their ABS SA2 district** (Victoria has 522), using only coordinates that fall
inside the district's boundary. The other 34 are kept and recorded, not guessed. Each listing has 111 columns: the
home itself, its SA2 district and 20 area features (population, income and projected growth), and its distance to
transport and services.

## 2. Our approach

| Step | Notebook | What it does |
|---|---|---|
| Collect | `02_scraping` | Scrapes rent.com.au for the 654 suburbs in the course data. The suburbs were split evenly between the five of us, and the script can be stopped and restarted |
| Clean | `01_data_curation` | One set of cleaning rules (`scripts/listings.py`) for both years and both websites. Odd values are flagged and set aside, never deleted |
| Area features | `04_abs_sa2_crosswalk` | Places each listing in the ABS SA2 district its coordinates fall inside, and adds 20 area features: population 2021–25, personal income 2018–19 to 2022–23 and VIF2023 projections. Also builds the suburb → SA2 lookup that links DFFH suburbs to population in Q2 |
| Location features | `05_geospatial_routing` | Distances to transport, schools, parks, shops, hospitals and entertainment, and the drive to the CBD and nearest station |
| **Q1** | `06_rent_drivers` | Seven models compared on areas they have not seen. Importance is how much worse the best model gets without a feature |
| **Q2** | `03_dffh_panel` | Six forecasting methods tested on five past periods. The most accurate one forecasts every DFFH area to 2030 |
| **Q3** | `07_liveability` | Liveability follows the published Urban Liveability Index. Affordability is annual rent ÷ area income |

**Three choices shape the results:**
- **Rent growth comes from 25 years of government rent data, not from our listings.** Two listing snapshots a year
  apart cannot support a five-year forecast. The DFFH series also records rents actually agreed, not asking prices.
- **Models are tested on areas they have never seen.** Nearby homes have similar rents, so testing on random
  listings would flatter the model. We hold out whole SA2 districts instead.
- **We use published methods where they exist.** For example, liveability follows the Urban Liveability Index
  (Higgs et al., 2019) instead of a formula we invented.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))   # shared project code

import pandas as pd
import config

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

## 3. Question 1: what drives rent?

We fitted seven models, from a simple rule of thumb to gradient boosting. Each one predicts log weekly rent from 35
features of the home and its location, and is scored on SA2 districts it has not seen. **Gradient boosting**
predicted best: on held-out test areas it is about **\$92 a week** off on average, with a typical error of 9.5% of
the rent. The baseline is about \$141.

In [2]:
models = pd.read_csv(config.CURATED / "q1_model_comparison.csv")
models[["model", "validation MAE ($/wk)", "test MAE ($/wk)", "test median error (%)", "test R2 (log rent)"]].round(2)

,model,validation MAE ($/wk),test MAE ($/wk),test median error (%),test R2 (log rent)
0,Gradient boosting,91.68,91.72,9.53,0.68
1,Random forest,94.18,92.10,8.93,0.67
2,Linear regression,103.64,106.08,11.44,0.59
3,Ridge,103.66,106.08,11.44,0.59
4,Lasso,103.80,106.01,11.41,0.59
5,Decision tree,106.64,105.76,10.60,0.58
6,Baseline,138.03,140.80,15.00,0.26


**Importance** is how much the model's error rises when a group of features is taken away. The range comes from
two ways of taking it away: removing the group and refitting, or shuffling its values.

![Importance of each feature group](../plots/q1_group_importance.png)

- **Size** (bedrooms, bathrooms, car spaces) matters most: the error rises by 40–146% without it.
- **City access** comes next (10–61%), then **area income and growth** (4–28%) and **dining and entertainment
  nearby** (2–16%).
- Outdoor features and extras add 4–9%. Property type, appliances, local services and public transport add little
  once the rest is known (under 3% each).

A feature is called a **robust** driver only if it matters in all four settings: the best model, the best linear
model, 2025 listings only and 2026 listings only. The pricing guide below shows what each home feature is worth in
dollars, for Melbourne and for regional Victoria.

In [3]:
guide = pd.read_csv(config.CURATED / "q1_pricing_guide.csv").rename(columns={"Unnamed: 0": "feature"})
guide = guide[guide["driver"].isin(["robust", "likely"])]
guide[["feature", "change", "metro $/wk", "regional $/wk", "driver", "who can act"]].reset_index(drop=True)

,feature,change,metro $/wk,regional $/wk,driver,who can act
0,Bedrooms,one more,98.0,46.0,robust,fixed
1,Bathrooms,one more,81.0,65.0,robust,fixed
2,Pool,without → with,70.0,80.0,likely,agent should mention
3,Furnished,without → with,48.0,46.0,robust,landlord could add
4,Study,without → with,29.0,18.0,robust,agent should mention
5,Car spaces,one more,22.0,7.0,robust,fixed
6,Dishwasher,without → with,20.0,20.0,robust,landlord could add
7,Outdoor entertaining,without → with,14.0,11.0,likely,agent should mention
8,Balcony deck,without → with,14.0,17.0,robust,agent should mention
9,Secure parking,without → with,8.0,5.0,likely,agent should mention


**A first look by area agrees (`04`).** Across SA2 districts, median rent rises with area income (r = 0.43 in 2025,
0.38 in 2026). Projected population growth has almost no link with current rent (r = −0.04 and −0.09): the
fastest-growing areas are outer suburbs with mid-range rents.

**Robust drivers outside the home:** the drive to the CBD, the area's median income, dining and entertainment within
1 km, being in Greater Melbourne, and the distance to the nearest hospital and park.

**Not reliable:** air conditioning, a courtyard, a shed, broadband and a gym. They look valuable in some settings
but not in all four.

## 4. Question 2: where will rents grow fastest by 2030?

We tested six forecasting methods by pretending to stand at five past dates (2012 to 2020) and checking how well
each method predicted the next five years. A **blend** of methods was most accurate: about 10% off after five years,
against 17% for assuming rents stay the same. That method forecasts all 145 DFFH areas to 2030Q3. Each DFFH area is
a group of one to four suburbs.

In [4]:
forecast = pd.read_csv(config.CURATED / "rent_forecast_areas.csv")
top10 = forecast.nlargest(10, "growth_pa_pct").assign(where=lambda d: d["metro"].map({True: "Melbourne", False: "Regional"}))
top10[["area", "where", "rent_now", "rent_5y", "growth_pa_pct", "growth_pa_low", "growth_pa_high"]].round(1).reset_index(drop=True)

,area,where,rent_now,rent_5y,growth_pa_pct,growth_pa_low,growth_pa_high
0,Swan Hill,Regional,430.0,638.8,8.2,5.9,13.1
1,Moe-Newborough,Regional,390.0,572.8,8.0,5.7,12.8
2,Warragul,Regional,550.0,803.0,7.9,5.6,12.7
3,Morwell,Regional,380.0,540.9,7.3,5.0,12.1
4,Mildura,Regional,460.0,649.6,7.1,4.8,12.0
5,Shepparton,Regional,460.0,645.8,7.0,4.7,11.8
6,Wodonga,Regional,500.0,699.0,6.9,4.6,11.7
7,Frankston,Melbourne,550.0,756.1,6.6,4.3,11.4
8,Croydon-Lilydale,Melbourne,600.0,821.3,6.5,4.2,11.3
9,Benalla,Regional,450.0,613.6,6.4,4.1,11.2


![Top 10 areas by forecast rent growth](../plots/forecast_top10.png)

- **Regional centres dominate** because they start from much lower rents, so the same dollar rise is a bigger
  percentage. Within Melbourne alone, Frankston, Croydon–Lilydale and the Yarra Ranges lead.
- **Treat the top 10 as a group, not an exact order.** The uncertainty ranges of neighbouring areas overlap.
- **Population growth did not help.** Adding it did not make any method more accurate, so the forecast uses rent
  history only. A likely reason is that the fastest-growing areas are new estates, where homes are built as people
  arrive. This matches `04`, where projected population growth has almost no link with current rent levels.

## 5. Question 3: the most liveable and affordable suburbs

**Liveability** follows the Urban Liveability Index (Higgs et al., 2019). It covers seven domains from Australian
liveability research (Badland et al., 2014): transport, education, parks, food and shops, health, leisure and
culture, and access to jobs. Transport, schools, parks and shops are scored on walking distance to Victorian planning standards (e.g. a
park within 400 m, a train station within 800 m); health, leisure and jobs on the distance to the nearest hospital,
the number of cafes, restaurants and venues within 1 km, and the drive to the CBD. Each domain is rescaled so 100 is
the average, and the domains are equally weighted with a small penalty for uneven suburbs (so being strong in one
domain isn't enough). A suburb's score is the average over its homes (suburbs with at least 10 listings: 370 in
total).
**Affordability** is kept separate: median annual rent ÷ the area's median personal income (lower is more
affordable).

In [5]:
suburbs = pd.read_csv(config.CURATED / "suburb_liveability.csv")
cols = ["suburb", "liveability", "liveability_rank", "median_rent", "rent_to_income", "affordability_rank"]

both = suburbs[(suburbs["liveability"] >= suburbs["liveability"].median())
               & (suburbs["rent_to_income"] <= suburbs["rent_to_income"].median())]
for name, part in [("Greater Melbourne", both[both["metro"]]), ("Regional Victoria", both[~both["metro"]])]:
    print(f"{name}: {len(part)} suburbs are both more liveable and more affordable than the median")
    display(part.sort_values("liveability", ascending=False)[cols].head(5).round(2).reset_index(drop=True))

Greater Melbourne: 77 suburbs are both more liveable and more affordable than the median


,suburb,liveability,liveability_rank,median_rent,rent_to_income,affordability_rank
0,Fitzroy,109.13,4,600.0,0.44,66
1,Collingwood,108.95,5,550.0,0.36,11
2,East Melbourne,108.64,6,710.0,0.46,97
3,Brunswick,108.02,9,682.5,0.50,163
4,Travancore,107.72,10,530.0,0.45,85


Regional Victoria: 6 suburbs are both more liveable and more affordable than the median


,suburb,liveability,liveability_rank,median_rent,rent_to_income,affordability_rank
0,Geelong,102.76,71,525.0,0.46,102
1,Geelong West,101.75,105,450.0,0.36,10
2,Newtown,100.40,140,515.0,0.40,29
3,Bell Park,100.33,142,500.0,0.46,105
4,Newcomb,99.71,166,490.0,0.50,170


![Liveability against affordability](../plots/liveability_vs_affordability.png)

- **Most liveable:** inner Melbourne, with Melbourne CBD (110.7), Carlton (110.4), West Melbourne, Fitzroy and
  Collingwood. These suburbs have transport, shops, schools, parks and entertainment within walking distance.
- **Liveable and affordable:** **Collingwood** stands out (5th most liveable, 11th most affordable). In regional
  Victoria, all six such suburbs are in Geelong and Ballarat, led by **Geelong West**.
- **The score is robust.** The ranking barely changes with a plain average (Spearman 0.999) or with any one domain
  removed (0.978–0.995).

### Linking Q2 and Q3: where to invest

The two answers together give a shortlist of suburbs with above-median liveability **and** above-median forecast
rent growth. Fast growth on its own is risky: of the fastest-growing 10% of suburbs, 68% have below-median
liveability.

In [6]:
xwalk = pd.read_csv(config.DFFH_CROSSWALK).dropna(subset=["suburb_key"])
growth = xwalk.merge(forecast[["area", "growth_pa_pct"]], left_on="dffh_area", right_on="area").drop_duplicates("suburb_key")
invest = suburbs.drop(columns=["dffh_area", "growth_pa_pct"]).merge(growth[["suburb_key", "dffh_area", "growth_pa_pct"]], on="suburb_key")
buy = invest[(invest["liveability"] >= suburbs["liveability"].median())
             & (invest["growth_pa_pct"] >= invest["growth_pa_pct"].median())]
print(f"{len(buy)} suburbs are liveable and above the median forecast growth (of {len(invest)} with a forecast)")
buy.sort_values("growth_pa_pct", ascending=False)[["suburb", "liveability", "growth_pa_pct", "median_rent", "rent_to_income"]].head(10).round(2).reset_index(drop=True)

50 suburbs are liveable and above the median forecast growth (of 190 with a forecast)


,suburb,liveability,growth_pa_pct,median_rent,rent_to_income
0,Frankston,99.71,6.57,617.5,0.56
1,Croydon,100.10,6.48,600.0,0.53
2,Lilydale,99.59,6.48,580.0,0.50
3,Bendigo,100.05,6.33,595.0,0.58
4,Port Melbourne,103.12,6.04,807.5,0.50
5,Carlton North,106.47,5.84,925.0,0.69
6,Geelong West,101.75,4.84,450.0,0.36
7,Alphington,103.64,4.55,725.0,0.46
8,Fairfield,104.77,4.55,435.0,0.30
9,Newcomb,99.71,4.44,490.0,0.50


![Liveability against forecast rent growth](../plots/liveability_vs_growth.png)

- **Highest growth on the shortlist:** Frankston, Croydon, Lilydale and Bendigo (6.3–6.6% a year). They are only
  just above the liveability cut-off, so they are borderline.
- **Strongest on both counts:** Carlton North and Port Melbourne. Both have high rents.
- **Liveable, growing and affordable:** **Geelong West** (4.8% a year, \$450 a week) and **Fairfield** (4.6%,
  \$435, the most affordable on the shortlist).

**Weighted top five (`07`, section 6).** Scoring the 50 shortlisted suburbs 40% growth, 30% liveability and 30%
affordability (percentile scores) puts **Fairfield** first, followed by Heidelberg, Geelong West, Alphington and
Clifton Hill. The weights are a judgement that favours rental return, not a published method, so the order can change
with other weights.

![Weighted top five: growth, liveability and rent-to-income](../plots/investment_heatmap_rent_to_income.png)

## 6. What this means for the client

- **Pricing listings.** The model and pricing guide (Q1) give a fair asking rent for a new listing from its
  features and location, typically within about 10%. They also show which features are worth highlighting in the
  advert, and which upgrades a landlord could make (furnishing, a dishwasher).
- **Choosing where to grow.** The forecast (Q2) and the investment shortlist (Q2 + Q3) point to where rental income
  is likely to rise. Liveable suburbs may also be easier to keep tenanted, though we did not test this.
- **Next steps for a real product.** Refresh the listings and DFFH data every quarter, re-run the pipeline, and add
  purchase prices so that rent growth can be turned into a return on investment.

## 7. Issues we ran into

- **Messy course data.** The 2025 Domain data had "rents" of \$315,000–\$808,500, car spaces listed as homes at
  \$40 a week, a 50-bedroom apartment and an empty land-area column. We wrote explicit cleaning rules, and flagged
  and set aside odd rows instead of deleting them (`01`).
- **Two websites, two vocabularies.** Domain and rent.com.au word features differently (653 Domain labels). We
  matched a fixed list of 28 features by meaning, so both years get the same columns. 2026 listings more often have
  no feature list at all.
- **Scraping at scale.** 654 suburbs were split evenly between five people. The scraper could stop and resume
  without overlap, and listings found in two suburbs' searches were kept once (`02`).
- **Suburbs and SA2 districts don't line up.** 134 suburbs span more than one SA2, and the DFFH rent history only
  names suburbs. We assigned each suburb to the SA2 holding most of its listings (43 suburbs have under 60% there).
  Listings themselves always use their own coordinates (`04`).
- **Notebooks that need each other's outputs.** 03 uses 04's population lookup, and 04's coverage check uses 03's
  suburb list. Both outputs are committed, and 04 skips that check on a fresh run, so either can be run first.
- **Too many routing requests.** Driving distances for 21,000 homes would need 20,000+ requests. We routed from each
  suburb's centre instead (about 800 routes), kept per-home straight-line distances, and cached every route (`05`).
- **Size and location are tangled.** Inner suburbs have smaller homes *and* are close to the CBD, so a simple
  correlation credits one with the other's effect. We used one model with both kinds of feature, tested it on areas
  it had not seen, and checked every ranking with a linear model and on each year separately (`06`).
- **No forecast foresaw the 2022–24 surge.** Every method tested from 2020 under-forecast it. We report ranges, not
  just a single number (`03`).
- **Working on shared notebooks.** Several people editing the same notebook led to merge conflicts. We moved to
  one shared branch (`main`) and agreed who edits which notebook.

## 8. Assumptions and limitations

**Data**
- Listing rents are **asking rents**, not signed leases. The Q2 forecast uses agreed rents (DFFH bonds).
- The two snapshots come from **different websites** a year apart, so differences between them are not pure market
  change.
- There is **no floor area, age or condition** data. Listed features partly stand for the overall quality of a home.
- Area income is **personal** income for 2022–23, not household income. Student-heavy areas look less affordable
  and high-income areas more affordable.
- **Area features describe the SA2 as a whole**, not the household renting a home. They come from different years
  (income to 2022–23, population to 2025, projections from 2023), all on 2021 SA2 boundaries.
- Regional Victoria has **few listings**: many SA2s there have fewer than 5, so area results are less precise.

**Q1**
- Importance and dollar values are **associations, not causes**.
- Location factors we do not measure, such as the coast or a suburb's prestige, explain part of the remaining error.

**Q2**
- The forecast **continues each area's trend**. It is more reliable for how much rents grow overall (about 10% error)
  than for the exact order of areas.
- Forecasts are for **DFFH areas** (groups of one to four suburbs), in nominal dollars, with data ending in 2025Q3.
- Housing (dwelling) growth was not used: area data start in 2021 and are mostly projections.

**Q3**
- **Equal weights** are a judgement, and central suburbs score well on several domains at once.
- Distances are **straight-line**, and transport counts **stops, not how often services run**.
- Crime, safety, air quality and local jobs are not measured.
- "Liveable" and "above-median growth" are **median splits**, so suburbs near the line (Frankston, Lilydale) are
  borderline.
- Any park counts as open space; the published index only counts parks over 1.5 ha.
- The drive to the CBD is one of the seven domains, so regional suburbs score lower partly by design; regional
  suburbs are compared separately for this reason.
- A suburb's score averages the homes listed for rent, which may differ from all homes in the suburb.
- The 40/30/30 weights in the investment ranking are a judgement; different weights would reorder the top five.

## 9. Privacy, security and reproducibility

- **No listing-level data is in the repository.** The repository is public, and the Domain data may not be
  redistributed. Only suburb- and area-level summaries and charts are committed. Notebooks show summaries, not
  individual listings.
- **No API keys are needed or stored.** Routing uses OSRM, which needs no key. Any key file (e.g. `ors_config.env`)
  is kept out of Git.
- **Reproducible.** Every file path is in `scripts/config.py`, and shared code is in `scripts/`. Package versions are
  pinned in `requirements.txt`. The README lists where to put each raw file and the order to run the notebooks.
  Slow steps (routes, model results) are cached, so re-runs are quick.

## 10. Where to find the details

| Notebook | Contents |
|---|---|
| `01_data_curation` | Exploring the 2025 data and the cleaning rules; builds the master listings table |
| `02_scraping` | Planning, checking and merging the rent.com.au scrape |
| `03_dffh_panel` | **Q2**: back-test, 2030 forecast, top 10 and the what-if rent income |
| `04_abs_sa2_crosswalk` | SA2 districts, population, income and projections; SA2 maps |
| `05_geospatial_routing` | Distances to transport and services; driving distances |
| `06_rent_drivers` | **Q1**: seven-model comparison, importance, reliability and the pricing guide |
| `07_liveability` | **Q3**: liveability index, affordability, the investment shortlist and the weighted top five |